# RGB Field Image Analysis

This notebook demonstrates a simple computer-vision workflow for visualising spatial colour variation in a field image.

**Important:** This is not scientific NDVI. True NDVI requires red and near-infrared (NIR) reflectance bands. The calculation below uses red and blue RGB channels as an exploratory visual proxy only; it does not measure soil quality or plant health.

## Workflow
1. Upload a field image.
2. Read it with OpenCV.
3. Calculate `(R - B) / (R + B)` per pixel.
4. Normalise the result for display and apply a colour map.


In [ ]:
from google.colab import files
from IPython.display import Image, display
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

def calculate_rgb_red_blue_index(image: np.ndarray) -> np.ndarray:
    """Return the exploratory (R - B) / (R + B) index for a BGR image."""
    if image is None or image.ndim != 3 or image.shape[2] != 3:
        raise ValueError("Expected a valid three-channel colour image.")
    blue, green, red = cv2.split(image.astype(np.float32))
    denominator = red + blue
    result = np.zeros_like(denominator, dtype=np.float32)
    np.divide(red - blue, denominator, out=result, where=denominator != 0)
    return np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0)

uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload an image to continue.")

image_path = next(iter(uploaded))
image = cv2.imread(image_path, cv2.IMREAD_COLOR)
if image is None:
    raise ValueError("OpenCV could not read the uploaded file. Please use a valid image.")

index_map = calculate_rgb_red_blue_index(image)
display_map = cv2.normalize(index_map, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
colour_map = cv2.applyColorMap(display_map, cv2.COLORMAP_JET)

print(f"Image dimensions: {image.shape[1]} x {image.shape[0]}")
print(f"Index range: {index_map.min():.4f} to {index_map.max():.4f}")
print("Original image:")
display(Image(filename=image_path))
print("RGB red-blue index visualisation (exploratory only):")
cv2_imshow(colour_map)


## Interpretation and limitations

- The colour map shows relative variation in the chosen RGB index within this image; colours are not universal health classes.
- Results can change with lighting, shadows, camera processing, and image content.
- No NIR band, calibration, ground-truth validation, or soil measurements are used.
- Do not interpret this output as a soil-quality, nutrient, moisture, or crop-health diagnosis.

## Next steps

A stronger remote-sensing version could accept calibrated red and NIR bands, calculate true NDVI, document reflectance preprocessing, and validate results against field observations.
